# Notebook 06 — Quantization and Memory Trade-offs

Quantization is where model quality, memory, and throughput collide. This notebook keeps the trade-offs explicit so you can plan local and single-node deployments without hand-wavy rules.

## What you will build

- compare common quantization families with transparent approximations
- estimate both weight memory and KV cache memory
- map hardware constraints to practical quantization choices
- simulate quality, latency, and throughput trade-offs
- record a decision report you can reuse in later serving notebooks

## Design rule

Quantization is never free. It buys fit and speed by changing numeric precision and kernel behavior. The right question is not whether quantization is good. The right question is which loss is acceptable for the workload you are serving.

In [ ]:
# --- Systems Notebook Setup ---
!pip install -q "transformers>=4.51.0" accelerate torch numpy pandas matplotlib fastapi uvicorn pydantic httpx psutil

import asyncio
import json
import math
import random
import statistics
import threading
import time
from collections import Counter, defaultdict, deque
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 140)

try:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE_DIR = Path("/content/drive/MyDrive/models")
    ARTIFACT_ROOT = Path("/content/drive/MyDrive/castalia-systems")
except Exception:
    CACHE_DIR = Path("./models")
    ARTIFACT_ROOT = Path("./artifacts")

CACHE_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

def now_ms():
    return time.perf_counter() * 1000

def summarize(values):
    return {
        "count": len(values),
        "mean": statistics.mean(values) if values else 0.0,
        "median": statistics.median(values) if values else 0.0,
        "min": min(values) if values else 0.0,
        "max": max(values) if values else 0.0,
    }

print("✓ Systems notebook environment ready")
print("  Cache dir:", CACHE_DIR)
print("  Artifact root:", ARTIFACT_ROOT)

In [ ]:
random.seed(6)

ARTIFACT_DIR = ARTIFACT_ROOT / "06_quantization_and_memory_tradeoffs"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

def estimate_weight_gib(params_billion, bits_per_weight):
    return params_billion * 1_000_000_000 * bits_per_weight / 8 / (1024 ** 3)

def estimate_kv_gib(batch_size, context_tokens, hidden_size=4096, layers=32, bytes_per_value=2):
    total_bytes = batch_size * context_tokens * hidden_size * layers * 2 * bytes_per_value
    return total_bytes / (1024 ** 3)

print("Artifact directory:", ARTIFACT_DIR.resolve())

## Step 1: Compare common quantization profiles

The exact file size depends on format details, but rough bits-per-weight estimates already tell you which choices are realistic.

In [ ]:
quant_profiles = [
    {"quant": "Q3_K_M", "bits_per_weight": 3.60, "relative_quality": 0.90, "relative_speed": 1.30, "notes": "very compact, quality loss can be visible"},
    {"quant": "Q4_K_M", "bits_per_weight": 4.83, "relative_quality": 0.95, "relative_speed": 1.18, "notes": "strong default for local work"},
    {"quant": "Q5_K_M", "bits_per_weight": 5.52, "relative_quality": 0.97, "relative_speed": 1.08, "notes": "more headroom on reasoning-heavy tasks"},
    {"quant": "Q6_K", "bits_per_weight": 6.57, "relative_quality": 0.985, "relative_speed": 0.98, "notes": "closer to reference quality"},
    {"quant": "Q8_0", "bits_per_weight": 8.50, "relative_quality": 0.995, "relative_speed": 0.88, "notes": "heavy but still easier than fp16"},
    {"quant": "F16", "bits_per_weight": 16.0, "relative_quality": 1.00, "relative_speed": 0.62, "notes": "reference-style storage, expensive locally"},
]

quant_df = pd.DataFrame(quant_profiles)
quant_df

## Step 2: Estimate weight memory for several model sizes

These estimates let you ask the first deployment question quickly: which files can I even try on this machine?

In [ ]:
model_sizes_b = [3, 7, 8, 14]
weight_rows = []
for size_b in model_sizes_b:
    for profile in quant_profiles:
        weight_rows.append({
            "model_b": size_b,
            "quant": profile["quant"],
            "weight_gib_est": round(estimate_weight_gib(size_b, profile["bits_per_weight"]), 2),
            "relative_quality": profile["relative_quality"],
            "relative_speed": profile["relative_speed"],
        })

weight_df = pd.DataFrame(weight_rows)
weight_df

## Step 3: Separate weight memory from KV cache memory

Weight size is only part of the budget. Context length, batch size, and datatype determine how much memory the KV cache consumes during generation.

In [ ]:
kv_rows = []
for batch_size in [1, 2, 4, 8]:
    for context_tokens in [2048, 4096, 8192, 16384]:
        kv_rows.append({
            "batch_size": batch_size,
            "context_tokens": context_tokens,
            "kv_gib_fp16_est": round(estimate_kv_gib(batch_size, context_tokens), 2),
        })

kv_df = pd.DataFrame(kv_rows)
kv_df

## Step 4: Visualize how context and batch push memory up

This is the reason serving settings matter so much. A quantized model can fit comfortably at batch 1 and become impossible at a larger batch or longer context.

In [ ]:
kv_pivot = kv_df.pivot(index="context_tokens", columns="batch_size", values="kv_gib_fp16_est")
ax = kv_pivot.plot(figsize=(8, 4), marker="o", title="Estimated KV cache memory (GiB)")
ax.set_ylabel("GiB")
plt.tight_layout()

## Step 5: Test hardware scenarios

The next question is operational: which model and quantization fit on actual hardware once you reserve safety headroom?

In [ ]:
hardware_scenarios = [
    {"name": "cpu_laptop_16g", "ram_gib": 16, "vram_gib": 0, "reserve_gib": 4},
    {"name": "desktop_32g", "ram_gib": 32, "vram_gib": 0, "reserve_gib": 6},
    {"name": "single_gpu_12g", "ram_gib": 32, "vram_gib": 12, "reserve_gib": 2},
    {"name": "single_gpu_24g", "ram_gib": 64, "vram_gib": 24, "reserve_gib": 3},
]

scenario_rows = []
for scenario in hardware_scenarios:
    usable_ram = scenario["ram_gib"] - scenario["reserve_gib"]
    usable_vram = max(scenario["vram_gib"] - scenario["reserve_gib"], 0)
    for row in weight_df[weight_df["model_b"] == 8].to_dict("records"):
        scenario_rows.append({
            "scenario": scenario["name"],
            "quant": row["quant"],
            "fits_cpu": row["weight_gib_est"] < usable_ram,
            "fits_full_gpu": row["weight_gib_est"] < usable_vram,
            "weight_gib_est": row["weight_gib_est"],
        })

scenario_df = pd.DataFrame(scenario_rows)
scenario_df

## Step 6: Simulate workload-aware trade-offs

Not every workload values quality equally. Below we score several tasks against each quantization using small transparent assumptions rather than opaque benchmark claims.

In [ ]:
workloads = [
    {"task": "chat_support", "difficulty": 0.9, "prompt_tokens": 220, "output_tokens": 120, "quality_floor": 0.92},
    {"task": "code_explainer", "difficulty": 1.0, "prompt_tokens": 380, "output_tokens": 180, "quality_floor": 0.95},
    {"task": "structured_extraction", "difficulty": 0.85, "prompt_tokens": 260, "output_tokens": 80, "quality_floor": 0.94},
    {"task": "reasoning_trace", "difficulty": 1.15, "prompt_tokens": 520, "output_tokens": 220, "quality_floor": 0.97},
]

def simulate_quant_run(workload, profile):
    quality = max(0.0, min(1.0, profile["relative_quality"] - 0.03 * (workload["difficulty"] - 1.0)))
    prefill_tps = 3200 * profile["relative_speed"]
    decode_tps = 42 * profile["relative_speed"]
    latency_s = workload["prompt_tokens"] / prefill_tps + workload["output_tokens"] / decode_tps
    passes = quality >= workload["quality_floor"]
    return {
        "quality_est": round(quality, 3),
        "latency_s": round(latency_s, 3),
        "throughput_tps": round(workload["output_tokens"] / max(latency_s, 0.001), 1),
        "meets_quality_floor": passes,
    }

tradeoff_rows = []
for workload in workloads:
    for profile in quant_profiles:
        tradeoff_rows.append({
            "task": workload["task"],
            "quant": profile["quant"],
            **simulate_quant_run(workload, profile),
        })

tradeoff_df = pd.DataFrame(tradeoff_rows)
tradeoff_df

## Step 7: Plot the quality versus latency frontier

You want the left-upper corner: low latency and high quality. Real workloads land on a frontier, not a single winner.

In [ ]:
chart_df = tradeoff_df[tradeoff_df["task"] == "reasoning_trace"].copy()
ax = chart_df.plot.scatter(x="latency_s", y="quality_est", s=80, figsize=(6, 4), title="Reasoning task: quality vs latency")
for _, row in chart_df.iterrows():
    ax.annotate(row["quant"], (row["latency_s"], row["quality_est"]))
plt.tight_layout()

## Step 8: Study the batch and context interaction

Quantization helps the weights fit, but aggressive batching can still exhaust memory through KV growth. This table combines both pressures.

In [ ]:
interaction_rows = []
base_weight_gib = {row["quant"]: row["weight_gib_est"] for row in weight_df[weight_df["model_b"] == 8].to_dict("records")}
for quant_name, weight_gib in base_weight_gib.items():
    for batch_size in [1, 2, 4, 8]:
        for context_tokens in [2048, 4096, 8192]:
            total_gib = weight_gib + estimate_kv_gib(batch_size, context_tokens)
            interaction_rows.append({
                "quant": quant_name,
                "batch_size": batch_size,
                "context_tokens": context_tokens,
                "total_memory_gib_est": round(total_gib, 2),
            })

interaction_df = pd.DataFrame(interaction_rows)
interaction_df.head(12)

## Step 9: Build a simple decision helper

Decision helpers are useful because deployment constraints are usually stated in plain language: small GPU, large context, and no visible quality collapse.

In [ ]:
def choose_quant(max_weight_gib, minimum_quality, prefer_speed=True):
    candidates = [
        profile for profile in quant_profiles
        if estimate_weight_gib(8, profile["bits_per_weight"]) <= max_weight_gib
        and profile["relative_quality"] >= minimum_quality
    ]
    if not candidates:
        return None
    key = (lambda item: item["relative_speed"]) if prefer_speed else (lambda item: item["relative_quality"])
    return max(candidates, key=key)

decision_examples = [
    {"scenario": "12 GiB GPU target", "max_weight_gib": 8.5, "minimum_quality": 0.95, "prefer_speed": True},
    {"scenario": "CPU laptop target", "max_weight_gib": 5.5, "minimum_quality": 0.92, "prefer_speed": True},
    {"scenario": "quality-first desktop", "max_weight_gib": 12.0, "minimum_quality": 0.985, "prefer_speed": False},
]

decision_rows = []
for example in decision_examples:
    choice = choose_quant(example["max_weight_gib"], example["minimum_quality"], example["prefer_speed"])
    decision_rows.append({
        "scenario": example["scenario"],
        "choice": choice["quant"] if choice else "no fit",
        "speed_bias": example["prefer_speed"],
    })

decision_df = pd.DataFrame(decision_rows)
decision_df

## Step 10: Make the quality loss visible with a toy evaluation set

The best way to stay honest is to score outputs. Here we use a tiny synthetic evaluation set and let quantization slightly change task pass rates.

In [ ]:
toy_eval = [
    {"case": "policy_summary", "importance": 0.9},
    {"case": "json_extraction", "importance": 1.0},
    {"case": "reasoning_chain", "importance": 1.15},
    {"case": "code_patch_plan", "importance": 1.05},
    {"case": "incident_report", "importance": 0.95},
]

eval_rows = []
for profile in quant_profiles:
    passed = 0
    for case in toy_eval:
        score = profile["relative_quality"] - 0.04 * (case["importance"] - 1.0)
        passed += 1 if score >= 0.95 else 0
    eval_rows.append({
        "quant": profile["quant"],
        "cases_passed": passed,
        "pass_rate": round(passed / len(toy_eval), 2),
    })

eval_df = pd.DataFrame(eval_rows)
eval_df

## Step 11: Record what you would log in a real experiment

A useful quantization report captures memory, quality, and runtime. Without all three, later optimizations become guesswork.

In [ ]:
report = {
    "notebook": "06_quantization_and_memory_tradeoffs",
    "default_recommendation": "Q4_K_M for broad local experimentation, Q5_K_M when quality margin matters, Q8_0 or F16 only when memory is generous.",
    "weight_memory_examples": weight_df[weight_df["model_b"] == 8].to_dict("records"),
    "kv_examples": kv_df[kv_df["batch_size"].isin([1, 4])].to_dict("records"),
    "decision_examples": decision_rows,
}

report_path = ARTIFACT_DIR / "quantization_tradeoff_report.json"
report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
print("Wrote report:", report_path.resolve())

## Practical recommendations

- start with `Q4_K_M` unless you have a strong reason not to
- upgrade to `Q5_K_M` or `Q6_K` when evaluation shows quality sensitivity
- track KV cache growth whenever you increase context or batch size
- choose quantization with a workload in mind, not from generic internet rankings

In [ ]:
recommendation_rows = []
for scenario in hardware_scenarios:
    recommendation_rows.append({
        "scenario": scenario["name"],
        "recommended_quant": choose_quant(
            max_weight_gib=max(scenario["ram_gib"] * 0.6, scenario["vram_gib"] - 2),
            minimum_quality=0.95 if scenario["vram_gib"] else 0.92,
            prefer_speed=True,
        )["quant"],
    })

pd.DataFrame(recommendation_rows)

## Recap

Quantization changes what fits, how fast decode runs, and how much quality headroom remains. The right answer depends on workload, hardware, and context length together. The next notebook turns that reasoning into a local OpenAI-compatible serving layer.